## Data source

I started by querying the **SINASC (Live Birth Information System)** dataset using the `basedosdados` Python package, which connects directly to **Google BigQuery**.

**Base dos Dados** is a Brazilian open-data platform that organizes and standardizes thousands of public datasets, making them easier to access through SQL and Python.

Dataset: https://basedosdados.org/dataset/48ccef51-8207-40ee-af5b-134c8ac3fb8c?table=80359f9a-8189-4693-bdf7-ebf7be0d2fff

In [3]:
%pip install basedosdados

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 4.9 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 4.4 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.6/958.6 kB 2.6 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: loguru
    Found existing installation: loguru 0.6.0
    Uninstalling loguru-0.6.0:
      Successfully uninstalled loguru-0.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37/37 [basedosdados] [google-cloud-bigquery]nt]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pysus 2.3.0 requires loguru<0.7.0,>=0.6.0, but you have loguru 0.7.3 which is incompatible.
pysus 2.3.0 requires numpy<2,>=1.22, but you have numpy 2.5.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [6]:
import basedosdados as bd
import os
from dotenv import load_dotenv
load_dotenv()

True

##### The query retrieves all live births recorded in SINASC between **2004 and 2024** where the mother was **14 years old or younger**. 

#### Besides the year and place of birth, it also includes demographic information such as the mother's age, race/color, marital status, education, occupation, municipality and state of residence, number of previous children, and the father's age.

#### To make the data easier to analyze, the query joins SINASC with the Base dos Dados dictionaries and directory tables, replacing coded values with descriptive labels (e.g., race/color and marital status) and adding the names of states and municipalities.

In [7]:
import basedosdados as bd

billing_id = os.getenv("billing_id")

query = """
WITH 
dicionario_escolaridade_mae AS (
    SELECT
        chave AS chave_escolaridade_mae,
        valor AS descricao_escolaridade_mae
    FROM `basedosdados.br_ms_sinasc.dicionario`
    WHERE
        nome_coluna = 'escolaridade_mae'
        AND id_tabela = 'microdados'
),
dicionario_estado_civil_mae AS (
    SELECT
        chave AS chave_estado_civil_mae,
        valor AS descricao_estado_civil_mae
    FROM `basedosdados.br_ms_sinasc.dicionario`
    WHERE
        nome_coluna = 'estado_civil_mae'
        AND id_tabela = 'microdados'
),
dicionario_raca_cor_mae AS (
    SELECT
        chave AS chave_raca_cor_mae,
        valor AS descricao_raca_cor_mae
    FROM `basedosdados.br_ms_sinasc.dicionario`
    WHERE
        nome_coluna = 'raca_cor_mae'
        AND id_tabela = 'microdados'
)

SELECT
    dados.ano AS ano,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.id_municipio_nascimento AS id_municipio_nascimento,
    diretorio_id_municipio_nascimento.nome AS id_municipio_nascimento_nome,
    dados.quantidade_filhos_vivos AS quantidade_filhos_vivos,
    dados.quantidade_filhos_mortos AS quantidade_filhos_mortos,
    dados.id_uf_mae AS id_uf_mae,
    dados.id_municipio_mae AS id_municipio_mae,
    diretorio_id_municipio_mae.nome AS id_municipio_mae_nome,
    dados.idade_mae AS idade_mae,
    descricao_escolaridade_mae AS escolaridade_mae,
    descricao_estado_civil_mae AS estado_civil_mae,
    dados.ocupacao_mae AS ocupacao_mae,
    descricao_raca_cor_mae AS raca_cor_mae,
    dados.idade_pai AS idade_pai

FROM `basedosdados.br_ms_sinasc.microdados` AS dados

LEFT JOIN (
    SELECT DISTINCT sigla, nome
    FROM `basedosdados.br_bd_diretorios_brasil.uf`
) AS diretorio_sigla_uf
ON dados.sigla_uf = diretorio_sigla_uf.sigla

LEFT JOIN (
    SELECT DISTINCT id_municipio, nome
    FROM `basedosdados.br_bd_diretorios_brasil.municipio`
) AS diretorio_id_municipio_nascimento
ON dados.id_municipio_nascimento = diretorio_id_municipio_nascimento.id_municipio

LEFT JOIN (
    SELECT DISTINCT id_municipio, nome
    FROM `basedosdados.br_bd_diretorios_brasil.municipio`
) AS diretorio_id_municipio_mae
ON dados.id_municipio_mae = diretorio_id_municipio_mae.id_municipio

LEFT JOIN dicionario_escolaridade_mae
ON dados.escolaridade_mae = chave_escolaridade_mae

LEFT JOIN dicionario_estado_civil_mae
ON dados.estado_civil_mae = chave_estado_civil_mae

LEFT JOIN dicionario_raca_cor_mae
ON dados.raca_cor_mae = chave_raca_cor_mae

WHERE
    dados.ano BETWEEN 2004 AND 2024
    AND dados.idade_mae <= 14
"""

df = bd.read_sql(
    query=query,
    billing_project_id=billing_id
)

df.head()

KeyboardInterrupt: 

In [ ]:
df.to_csv("microdados_sinasc_2004_2024.csv", index=False)

#### #### Next, I queried the total number of live births by municipality and year (2004–2024). These counts serve as the denominator for calculating rates of births to girls aged 14 and younger per 1,000 live births, making it possible to compare municipalities and states regardless of their population size.

In [ ]:
query = """
SELECT
    dados.ano AS ano,
    dados.sigla_uf AS sigla_uf,
    dados.id_municipio_nascimento AS id_municipio_nascimento,
    municipio.nome AS municipio_nascimento_nome,
    COUNT(*) AS total_nascidos_vivos

FROM `basedosdados.br_ms_sinasc.microdados` AS dados

LEFT JOIN (
    SELECT DISTINCT id_municipio, nome
    FROM `basedosdados.br_bd_diretorios_brasil.municipio`
) AS municipio
ON dados.id_municipio_nascimento = municipio.id_municipio

WHERE
    dados.ano BETWEEN 2004 AND 2024
    AND dados.id_municipio_nascimento IS NOT NULL

GROUP BY
    ano,
    sigla_uf,
    id_municipio_nascimento,
    municipio_nascimento_nome

ORDER BY
    ano,
    sigla_uf,
    municipio_nascimento_nome
"""

total_nascidos_municipio = bd.read_sql(
    query=query,
    billing_project_id=billing_id
)

total_nascidos_municipio.head()

NameError: name 'billing_id' is not defined

#### The resulting dataset was exported as a CSV file for analysis in a second Jupyter notebook.

In [ ]:
total_nascidos_municipio.to_csv("total_nascidos_municipio_2004_2024.csv", index=False)